# Experimental Pixal3D CUDA · mapped image/camera inputs · Colab T4 · 512 + PBR
**Not yet validated by a full pretrained CUDA MV run.** This is separate from the historical single-image acceptance result.
Select a fresh **T4 GPU** runtime. Supply N consistent images and one camera mapping per image (orbit parameters or Blender/NeRF c2w matrices) in `transforms.json`. Framing must match the poses; this does not calibrate arbitrary photos.
The [input contract](https://github.com/syi0808/pixal3d-cuda/blob/main/docs/multiview.md) explains coordinates and the explicitly assumed orbit template.
DINOv3 access and an HF read token are required. Leave at least 40 GiB disk free for the dedicated MV weights. All flows retain 12 steps; geometry is 512, output PBR textures 1024 by default. No automatic precision/resolution reduction.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

def run_command(command):
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)

if "torch" in sys.modules:
    raise RuntimeError("Restart the runtime before installing PyTorch.")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
REFERENCE = "main"
ROOT = Path("/content/pixal3d-cuda-mv")
if not ROOT.exists():
    run_command(["git", "clone", "--depth", "1", "--branch", REFERENCE,
                 "https://github.com/syi0808/pixal3d-cuda.git", str(ROOT)])
os.chdir(ROOT)
print("Experimental source:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
run_command([sys.executable, "-m", "pip", "install", "torch==2.6.0", "torchvision==0.21.0",
             "--index-url", "https://download.pytorch.org/whl/cu124"])
run_command([sys.executable, "-m", "pip", "install", "-e", f"{ROOT}[dev,preprocess]"])

In [ ]:
import shutil
import torch

assert torch.cuda.is_available(), "Select a T4 GPU runtime."
assert "T4" in torch.cuda.get_device_name(), torch.cuda.get_device_name()
assert shutil.disk_usage("/content").free >= 40 * 1024**3, "Free at least 40 GiB."
print(subprocess.check_output(["nvidia-smi"], text=True))
run_command([sys.executable, "-m", "pixalcuda.cli", "doctor"])
run_command([sys.executable, "-m", "pytest", "-q"])

## Authenticate and download MV checkpoints
Use Colab Secret **HF_TOKEN** with notebook access. Never paste the token into code or outputs. This mode does not need MoGe. Downloads can take time; install/download time is not a warm inference benchmark.

In [ ]:
from huggingface_hub import login, notebook_login

try:
    from google.colab import userdata
    secret = userdata.get("HF_TOKEN")
except Exception:
    secret = None
if secret:
    login(token=secret, add_to_git_credential=False)
    del secret
else:
    notebook_login()
run_command([sys.executable, "-m", "pixalcuda.cli", "download", "--multiview"])
run_command([sys.executable, "scripts/verify_checkpoints.py", "--multiview"])

## Upload mapped images and transforms.json
Upload **N image files and transforms.json** (flat filenames), not a ZIP. Each image needs one indexed camera or input-order camera; orbit parameters and explicit c2w matrices are supported. There is no implicit four-view rig or four-image model limit.
In the next mapping cell, set `IMAGE_ORDER` to the exact filename order indexed by your camera JSON. If it stays empty, every frame must provide a matching `file_path`; those paths and optional indices determine the image/camera pairs. Image order is never guessed from alphabetic sorting or upload selection order.
Original framing/alpha is kept. Example orbits are assumptions, not calibration. Set `NO_REMBG=True` only for isolated/masked inputs. Resume needs identical input bytes, mappings, settings and source.

In [ ]:
import json
from google.colab import files
from PIL import Image
from IPython.display import display
from pixalcuda.multiview import read_mapped_views, read_views

VIEWS = Path("/content/pixal3d-posed-input")
VIEWS.mkdir(exist_ok=True)
uploaded = files.upload()
assert len(uploaded) >= 2 and "transforms.json" in uploaded, "Upload images and transforms.json."
UPLOADED_NAMES = [name for name in uploaded if name != "transforms.json"]
for name, payload in uploaded.items():
    if Path(name).name != name:
        raise ValueError("Use flat uploaded filenames.")
    (VIEWS / name).write_bytes(payload)
print("Uploaded image filenames:", UPLOADED_NAMES)
del uploaded, payload

In [ ]:
IMAGE_ORDER = []
CAMERA_METADATA = json.loads((VIEWS / "transforms.json").read_text())
if IMAGE_ORDER:
    if len(IMAGE_ORDER) != len(UPLOADED_NAMES) or set(IMAGE_ORDER) != set(UPLOADED_NAMES):
        raise ValueError("IMAGE_ORDER must list every uploaded image exactly once.")
    IMAGE_PATHS = [VIEWS / name for name in IMAGE_ORDER]
    manifest = read_mapped_views(IMAGE_PATHS, CAMERA_METADATA)
else:
    manifest = read_views(VIEWS)
    IMAGE_PATHS = list(manifest.paths)
    if sorted(path.name for path in IMAGE_PATHS) != sorted(UPLOADED_NAMES):
        raise ValueError("Map every uploaded image exactly once, or explicitly set IMAGE_ORDER.")
print(json.dumps(manifest.identity, indent=2))
for path in IMAGE_PATHS:
    with Image.open(path) as source:
        display(source.copy())
OUTPUT = ROOT / "outputs" / "t4-mapped"
NUM_VIEWS = len(manifest.paths)
NO_REMBG = False
RESUME = False

## Real T4 acceptance run
This records source/model identity, all view hashes/poses, stage timings, GPU memory, RSS, and actual PBR GLB validation. Token/voxel guards fail explicitly rather than silently lowering 512 geometry.
For a controlled one-view MV comparison set `NUM_VIEWS=1` and a **new** OUTPUT directory. Do not compare it to the separate single-image weights as if only view count changed.

In [ ]:
command = [sys.executable, "-u", "scripts/validate_colab_t4.py",
           "--images", *map(str, IMAGE_PATHS), "--transforms", str(VIEWS / "transforms.json"),
           "--num-views", str(NUM_VIEWS), "--output-dir", str(OUTPUT)]
if NO_REMBG:
    command.append("--no-rembg")
if RESUME:
    command.append("--resume")
run_command(command)

In [ ]:
from pixalcuda.export import validate_glb

GLB = OUTPUT / "run" / "result.glb"
print(json.dumps(json.loads((OUTPUT / "acceptance.json").read_text()), indent=2))
print(json.dumps(validate_glb(GLB), indent=2))
report = json.loads((OUTPUT / "run" / "report.json").read_text())
for stage in report["stages"]:
    print(stage["name"], round(stage["elapsed_seconds"], 2),
          "CUDA allocated GiB", round(stage.get("peak_allocated_bytes", 0) / 1024**3, 3),
          "RSS GiB", round(stage["peak_sampled_rss_bytes"] / 1024**3, 3))

## Download actual GLB and evidence
Inspect the real GLB in a PBR viewer before accepting geometry/texture quality. The ZIP includes inputs and intermediate tensors: treat it as private. Nothing here publishes your assets.

In [ ]:
files.download(str(GLB))
archive = shutil.make_archive("/content/pixal3d-mv-t4-result", "zip", root_dir=OUTPUT)
files.download(archive)